In [4]:
from pathlib import Path
import sys
import os
import json
sys.path.append(os.path.abspath('..'))
import src.data.transformators as T
from src.utils import load_database, save_dataset
from scipy.signal import savgol_filter

In [ ]:
pipeline = T.Pipeline([
        T.TemporalJumpCutter(100),
        T.Interpolator('linear'),
        T.FrameNaNCutter(),
        T.LowConfidenceCutter(0.3),
        T.Smoother(lambda x: savgol_filter(x, window_length=15, polyorder=2)),
        T.KeypointCutter([0, 1, 2, 3, 4, 13, 14, 15, 16]),
        T.VirtualKeypoint([[0,1]]), #Neck -> 8
        T.VirtualKeypoint([[6,7]]), #MidHip -> 9
        T.ScoresCutter(),
        T.SkeletonCentering(midhip_index=9),
        T.HeightScaler(strategy=None, neck_idx=8, midhip_idx=9),
        T.KeypointCutter([8,9]),
        T.Sampler(60, 'last')
        ])

In [ ]:
dataset = load_database('../dataset/filtred/dataset.pkl')
dataset_preprocessed = pipeline.run(dataset)
save_dataset('../dataset/preprocessed/', dataset_preprocessed)